# Propensity Score Modeling

## Objective

This notebook estimates treatment assignment probabilities for the Hillstrom email marketing experiment.

The propensity score represents the probability that a customer receives a particular treatment given their observed pre-treatment characteristics.

The objectives are:

- Prepare the treatment-model dataset.
- Use the covariates defined in the causal formulation.
- Estimate multi-class treatment probabilities.
- Estimate pairwise propensity scores for:
  - Mens E-Mail vs No E-Mail
  - Womens E-Mail vs No E-Mail
- Check treatment probability distributions.
- Check positivity/common support.
- Save the fitted propensity models for downstream causal modeling.

The estimated propensity scores will be used as nuisance information in the downstream Double Machine Learning pipeline.


In [1]:
import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger(__name__)

logger.info("Libraries imported successfully.")

2026-09-25 16:03:17 [INFO] Libraries imported successfully.


## Treatment and Covariate Definition

The treatment variable is encoded as:

- `0` → No E-Mail
- `1` → Mens E-Mail
- `2` → Womens E-Mail

The treatment model uses only pre-treatment customer characteristics.

The outcome variables (`visit`, `conversion`, and `spend`) are not used as predictors of treatment assignment because they occur after treatment and would introduce post-treatment information into the treatment model.

In [2]:
DATA_PATHS = [
    Path("../Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"),
    Path("Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"),
    Path("../datasets/raw/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")
]

dataset_path = next((path for path in DATA_PATHS if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Hillstrom dataset CSV not found. Please verify the project directory."
    )

df = pd.read_csv(dataset_path)

df = df.drop_duplicates().reset_index(drop=True)

logger.info(f"Dataset loaded successfully: {df.shape}")
logger.info(f"Remaining duplicate rows: {df.duplicated().sum()}")

2026-09-25 16:03:40 [INFO] Dataset loaded successfully: (57438, 12)
2026-09-25 16:03:40 [INFO] Remaining duplicate rows: 0


In [3]:
TREATMENT_COL = "segment"

COVARIATE_COLS = [
    "recency",
    "history_segment",
    "history",
    "mens",
    "womens",
    "zip_code",
    "newbie",
    "channel"
]

TREATMENT_MAPPING = {
    "No E-Mail": 0,
    "Mens E-Mail": 1,
    "Womens E-Mail": 2
}

df["treatment"] = df[TREATMENT_COL].map(TREATMENT_MAPPING)

if df["treatment"].isnull().any():
    unknown_values = df.loc[
        df["treatment"].isnull(),
        TREATMENT_COL
    ].unique()

    raise ValueError(
        f"Unknown treatment categories found: {unknown_values}"
    )

logger.info("Treatment encoding completed.")

2026-09-25 16:04:00 [INFO] Treatment encoding completed.


## Propensity Model Preprocessing

The treatment model contains both numerical and categorical customer characteristics.

Numerical variables:

- `recency`
- `history`

Binary variables:

- `mens`
- `womens`
- `newbie`

Categorical variables:

- `zip_code`
- `channel`
- `history_segment`

The preprocessing follows the feature structure established in Notebook 2.

In [4]:
NUMERIC_FEATURES = [
    "recency",
    "history"
]

BINARY_FEATURES = [
    "mens",
    "womens",
    "newbie"
]

CATEGORICAL_FEATURES = [
    "zip_code",
    "channel",
    "history_segment"
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

binary_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

propensity_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("binary", binary_pipeline, BINARY_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES)
    ],
    remainder="drop"
)

logger.info("Propensity-score preprocessing pipeline created.")

2026-09-25 16:04:28 [INFO] Propensity-score preprocessing pipeline created.


In [5]:
X_treatment = df[COVARIATE_COLS].copy()
T = df["treatment"].copy()

logger.info(f"Treatment-model feature shape: {X_treatment.shape}")
logger.info(f"Treatment vector shape: {T.shape}")

print(T.value_counts().sort_index())

2026-09-25 16:04:48 [INFO] Treatment-model feature shape: (57438, 8)
2026-09-25 16:04:48 [INFO] Treatment vector shape: (57438,)


treatment
0    19081
1    19183
2    19174
Name: count, dtype: int64


## Multi-Class Propensity Score Model

Because the experiment contains three treatment arms, the treatment model estimates:

$$
P(T=0|X)
$$

$$
P(T=1|X)
$$

$$
P(T=2|X)
$$

where:

- `T=0` is No E-Mail
- `T=1` is Mens E-Mail
- `T=2` is Womens E-Mail

The three probabilities for each customer should sum to approximately one.

In [8]:
multiclass_propensity_model = Pipeline(
    steps=[
        ("preprocessor", propensity_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

multiclass_propensity_model.fit(X_treatment, T)

multiclass_propensity = (
    multiclass_propensity_model
    .predict_proba(X_treatment)
)

logger.info("Multi-class propensity model fitted successfully.")

multiclass_propensity.shape

2026-09-25 16:05:57 [INFO] Multi-class propensity model fitted successfully.


(57438, 3)

In [9]:
propensity_columns = [
    "propensity_no_email",
    "propensity_mens_email",
    "propensity_womens_email"
]

propensity_df = pd.DataFrame(
    multiclass_propensity,
    columns=propensity_columns,
    index=df.index
)

propensity_df.head()

,propensity_no_email,propensity_mens_email,propensity_womens_email
0,0.341207,0.326631,0.332162
1,0.325717,0.344416,0.329866
2,0.338565,0.328090,0.333345
3,0.327500,0.332749,0.339751
4,0.331760,0.344649,0.323591


In [10]:
propensity_df["probability_sum"] = propensity_df.sum(axis=1)

propensity_df["probability_sum"].describe()

count    5.743800e+04
mean     1.000000e+00
std      5.551647e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: probability_sum, dtype: float64

## Pairwise Propensity Models

For downstream causal comparisons, two binary treatment models are constructed:

### Mens E-Mail vs No E-Mail

- `0` → No E-Mail
- `1` → Mens E-Mail

### Womens E-Mail vs No E-Mail

- `0` → No E-Mail
- `1` → Womens E-Mail

These pairwise models provide treatment probabilities specifically for each campaign comparison.

In [11]:
mens_mask = df["treatment"].isin([0, 1])

mens_X = df.loc[mens_mask, COVARIATE_COLS].copy()
mens_T = (df.loc[mens_mask, "treatment"] == 1).astype(int)

womens_mask = df["treatment"].isin([0, 2])

womens_X = df.loc[womens_mask, COVARIATE_COLS].copy()
womens_T = (df.loc[womens_mask, "treatment"] == 2).astype(int)

logger.info(f"Mens vs Control shape: {mens_X.shape}")
logger.info(f"Womens vs Control shape: {womens_X.shape}")

2026-09-25 16:07:17 [INFO] Mens vs Control shape: (38264, 8)
2026-09-25 16:07:17 [INFO] Womens vs Control shape: (38255, 8)


In [12]:
mens_propensity_model = Pipeline(
    steps=[
        ("preprocessor", propensity_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

mens_propensity_model.fit(mens_X, mens_T)

mens_propensity = mens_propensity_model.predict_proba(
    mens_X
)[:, 1]

logger.info("Mens vs Control propensity model fitted.")

2026-09-25 16:07:27 [INFO] Mens vs Control propensity model fitted.


In [13]:
womens_propensity_model = Pipeline(
    steps=[
        ("preprocessor", propensity_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

womens_propensity_model.fit(womens_X, womens_T)

womens_propensity = womens_propensity_model.predict_proba(
    womens_X
)[:, 1]

logger.info("Womens vs Control propensity model fitted.")

2026-09-25 16:07:40 [INFO] Womens vs Control propensity model fitted.


## Positivity and Common Support

A key causal assumption is positivity.

For the relevant customer profiles, the probability of receiving each treatment should not be effectively zero.

Very extreme propensity scores can indicate limited overlap between treatment groups.

We therefore inspect the minimum, maximum, mean, and quantiles of the estimated treatment probabilities.

In [14]:
propensity_summary = propensity_df[
    propensity_columns
].describe().T

propensity_summary

,count,mean,std,min,25%,50%,75%,max
propensity_no_email,57438.0,0.332194,0.006280,0.294847,0.327657,0.331478,0.337225,0.347805
propensity_mens_email,57438.0,0.333971,0.008698,0.313792,0.327394,0.333200,0.339881,0.369009
propensity_womens_email,57438.0,0.333835,0.007434,0.305270,0.328659,0.333192,0.338469,0.356175


In [15]:
pairwise_summary = pd.DataFrame({
    "Mens_vs_Control": pd.Series(mens_propensity).describe(),
    "Womens_vs_Control": pd.Series(womens_propensity).describe()
})

pairwise_summary

,Mens_vs_Control,Womens_vs_Control
count,38264.000000,38255.000000
mean,0.501269,0.501217
std,0.010011,0.007889
min,0.477438,0.475376
25%,0.493838,0.495766
50%,0.501171,0.500366
75%,0.508402,0.506543
max,0.547156,0.542360


In [16]:
extreme_threshold = 0.01

extreme_counts = {
    "Mens_vs_Control_below_0.01": int(
        (mens_propensity < extreme_threshold).sum()
    ),
    "Mens_vs_Control_above_0.99": int(
        (mens_propensity > 1 - extreme_threshold).sum()
    ),
    "Womens_vs_Control_below_0.01": int(
        (womens_propensity < extreme_threshold).sum()
    ),
    "Womens_vs_Control_above_0.99": int(
        (womens_propensity > 1 - extreme_threshold).sum()
    )
}

pd.Series(extreme_counts)

Mens_vs_Control_below_0.01      0
Mens_vs_Control_above_0.99      0
Womens_vs_Control_below_0.01    0
Womens_vs_Control_above_0.99    0
dtype: int64

## Propensity Score Interpretation

The propensity score is a model-based estimate of treatment assignment probability given the observed customer characteristics.

Because the Hillstrom dataset is a randomized experiment, the propensity scores are not required to recover the original experimental treatment effect. They provide useful nuisance information for the project's general causal-learning architecture and for downstream heterogeneous-effect modeling.

The pairwise scores are particularly useful when comparing each marketing campaign against the No E-Mail control group.

In [17]:
ARTIFACT_DIR = Path("../models")

if not ARTIFACT_DIR.exists():
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    multiclass_propensity_model,
    ARTIFACT_DIR / "multiclass_propensity_model.pkl"
)

joblib.dump(
    mens_propensity_model,
    ARTIFACT_DIR / "mens_propensity_model.pkl"
)

joblib.dump(
    womens_propensity_model,
    ARTIFACT_DIR / "womens_propensity_model.pkl"
)

logger.info("Propensity models saved successfully.")

2026-09-25 16:09:02 [INFO] Propensity models saved successfully.


In [18]:
propensity_output = df[
    ["treatment"]
].copy()

propensity_output = pd.concat(
    [
        propensity_output,
        propensity_df[propensity_columns]
    ],
    axis=1
)

propensity_output.head()

,treatment,propensity_no_email,propensity_mens_email,propensity_womens_email
0,2,0.341207,0.326631,0.332162
1,0,0.325717,0.344416,0.329866
2,2,0.338565,0.328090,0.333345
3,1,0.327500,0.332749,0.339751
4,2,0.331760,0.344649,0.323591


# Conclusion

The treatment assignment models have been established.

### Completed

- Treatment encoding
- Pre-treatment covariate preparation
- Multi-class propensity modeling
- Mens E-Mail vs No E-Mail propensity modeling
- Womens E-Mail vs No E-Mail propensity modeling
- Propensity probability diagnostics
- Positivity/common-support checks
- Model artifact saving

### Next Step

The estimated treatment probabilities will be used as nuisance information in the downstream Double Machine Learning stage.

The next notebook will focus on outcome modeling, treatment residualization, and Double Machine Learning estimation.